# LoRA v7 — Clean Chat-Template Training

- Uses SmolVLM's built-in chat template instead of raw prompts
- SFTTrainer handles label masking, padding, and training loop automatically
- Native resolution images (no resize)
- Trains the model to output just the letter in the assistant role
- Simple model.generate() at inference — aligned with training
- 1 epoch, same LoRA config as v2
- No custom collator debugging, no tokenizer issues

## Cell 0: Install

In [1]:
!pip install -q "transformers==4.57.1" "peft==0.18.0" "trl==0.18.1" bitsandbytes accelerate "pandas==2.2.2" "pillow==11.3.0" tqdm

## Cell 1: Download Data

In [2]:
import kagglehub
from pathlib import Path
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
COMP_DIR = Path(kagglehub.competition_download("pixels-to-predictions"))
print("Downloaded to:", COMP_DIR)

Downloaded to: /root/.cache/kagglehub/competitions/pixels-to-predictions


## Cell 2: Imports & Config

In [4]:
import os, json, random, gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

SEED = 42; random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
os.environ["WANDB_DISABLED"] = "true"

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
CHOICE_LETTERS = "ABCDEFGHIJ"

from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = Path("/content/lora_v7")
DRIVE_SAVE = Path("/content/drive/MyDrive/lora_v7_clean")

print(f"GPU: {torch.cuda.is_available()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: True


## Cell 3: Load Data

In [5]:
DATA_DIR = COMP_DIR / "data"
if not DATA_DIR.exists(): DATA_DIR = COMP_DIR

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

example_name = Path(val_df.iloc[0]["image_path"]).name
matches = list(COMP_DIR.rglob(example_name))
DATA_ROOT = matches[0].parents[2]
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 3109 | Val: 1048 | Test: 1008


## Cell 4: Load Model + Processor

In [6]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, attn_implementation="eager",
)

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Base model loaded. Will apply LoRA during training.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Base model loaded. Will apply LoRA during training.


## Cell 5: Build Chat-Template Dataset

In [7]:
def clean_text(x):
    if pd.isna(x): return ""
    return str(x).strip()

def load_image(row):
    path = DATA_ROOT / row["image_path"]
    img = Image.open(path).convert("RGB")
    img.thumbnail((512, 512), Image.BICUBIC)
    return img

def build_chat_messages(row, include_answer=True):
    choices = row["choices"]
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))

    user_text = ""
    if lecture: user_text += f"Lecture: {lecture}\n\n"
    if hint: user_text += f"Hint: {hint}\n\n"
    user_text += f"Question: {question}\n\n"
    user_text += f"Choices:\n{choices_text}\n\n"
    user_text += "Answer with just the letter (A, B, C, etc)."

    user_content = [
        {"type": "image"},
        {"type": "text", "text": user_text},
    ]

    messages = [{"role": "user", "content": user_content}]

    if include_answer:
        ans = int(row["answer"])
        letter = CHOICE_LETTERS[ans]
        messages.append({"role": "assistant", "content": [{"type": "text", "text": letter}]})

    return messages

# Test chat template
row = train_df.iloc[0]
msgs = build_chat_messages(row, include_answer=True)
text = processor.apply_chat_template(msgs, tokenize=False)
print(text[:300])
print("...")
print(text[-100:])

<|im_start|>User:<image>Lecture: Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For exa
...
frogs

Answer with just the letter (A, B, C, etc).<end_of_utterance>
Assistant: C<end_of_utterance>



## Cell 6: Prepare Dataset for SFTTrainer

In [8]:
from torch.utils.data import Dataset

class ChatVQADataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = load_image(row)
        messages = build_chat_messages(row, include_answer=True)
        text = processor.apply_chat_template(messages, tokenize=False)
        return {"images": [image], "texts": text}

def collate_fn(examples):
    texts = [e["texts"] for e in examples]
    images = [e["images"][0] for e in examples]
    batch = processor(text=texts, images=images, return_tensors="pt", padding=True, truncation=False)

    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100

    # Mask everything except assistant response
    for i in range(len(examples)):
        # Find where assistant response starts
        # The assistant token varies by model, find it
        input_ids = batch["input_ids"][i]
        # Tokenize just the user part (without assistant response)
        user_msgs = [{"role": "user", "content": build_chat_messages(train_df.iloc[0])[0]["content"]}]
        # Simpler approach: find the answer token and only train on that
        # The answer is always the last few tokens before padding
        seq_len = batch["attention_mask"][i].sum().item()
        # Mask all but last 2 tokens (the letter + possible EOS)
        labels[i, :seq_len - 2] = -100

    batch["labels"] = labels
    return batch

# Test
train_dataset = ChatVQADataset(train_df)
sample = train_dataset[0]
print(f"Sample text length: {len(sample['texts'])}")
print(f"Image size: {sample['images'][0].size}")

Sample text length: 2616
Image size: (302, 252)


## Cell 7: Train with SFTTrainer

In [9]:
processor.tokenizer.padding_side = "right"  # right padding for training

training_args = SFTConfig(
    output_dir=str(SAVE_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=3e-5,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=50,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=1.0,
    seed=SEED,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    dataset_text_field="texts",
    dataset_kwargs={"skip_prepare_dataset": True},
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    peft_config=lora_config,
    processing_class=processor.tokenizer,
)

trainer.model.print_trainable_parameters()
trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,}")
assert trainable <= 5_000_000, f"Over limit: {trainable:,}"

trainer.train()

# Save
trainer.save_model(str(SAVE_DIR))
processor.save_pretrained(str(SAVE_DIR))
import shutil
DRIVE_SAVE.mkdir(parents=True, exist_ok=True)
if DRIVE_SAVE.exists(): shutil.rmtree(DRIVE_SAVE)
shutil.copytree(SAVE_DIR, DRIVE_SAVE)
print(f"Saved to {DRIVE_SAVE}")

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 49279, 'bos_token_id': 1, 'pad_token_id': 2}.


trainable params: 4,161,536 || all params: 511,643,840 || trainable%: 0.8134
Trainable: 4,161,536


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
50,5.702400
100,0.079100
150,0.000700


Saved to /content/drive/MyDrive/lora_v7_clean


## Cell 8: Load for Inference

In [10]:
del model, trainer; gc.collect(); torch.cuda.empty_cache()

processor.tokenizer.padding_side = "left"  # left padding for inference

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, attn_implementation="eager",
)
model = PeftModel.from_pretrained(model, str(DRIVE_SAVE))
model.eval()
print("Loaded v7 checkpoint")

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Loaded v7 checkpoint


## Cell 9: Inference with Generate

In [11]:
BATCH_SIZE_INFER = 4

def predict_batch(df_batch):
    images = [load_image(row) for _, row in df_batch.iterrows()]
    messages_list = [build_chat_messages(row, include_answer=False) for _, row in df_batch.iterrows()]
    texts = [processor.apply_chat_template(m, add_generation_prompt=True, tokenize=False) for m in messages_list]

    inputs = processor(text=texts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=3, do_sample=False)

    input_len = inputs["input_ids"].shape[1]
    preds = []
    for i, (_, row) in enumerate(df_batch.iterrows()):
        generated = processor.tokenizer.decode(outputs[i, input_len:], skip_special_tokens=True).strip()
        letter = generated[0] if generated and generated[0] in CHOICE_LETTERS else None
        if letter and letter in CHOICE_LETTERS[:len(row["choices"])]:
            preds.append(CHOICE_LETTERS.index(letter))
        else:
            preds.append(0)
    return preds

## Cell 10: Validate

In [12]:
model.eval(); gc.collect(); torch.cuda.empty_cache()
eval_df = val_df.copy().reset_index(drop=True)
print(f"Full validation: {len(eval_df)} samples")

all_preds = []
for start in tqdm(range(0, len(eval_df), BATCH_SIZE_INFER), desc="Val"):
    batch = eval_df.iloc[start:start+BATCH_SIZE_INFER]
    try:
        p = predict_batch(batch)
    except RuntimeError:
        torch.cuda.empty_cache()
        p = []
        for j in range(len(batch)):
            p.extend(predict_batch(batch.iloc[j:j+1]))
            torch.cuda.empty_cache()
    all_preds.extend(p)
    torch.cuda.empty_cache()

eval_df["pred"] = all_preds
acc = (eval_df["pred"] == eval_df["answer"]).mean()
print(f"\nFull val accuracy: {acc:.4f}")
print(f"Previous best leaderboard: 0.768")

Full validation: 1048 samples


Val:   0%|          | 0/262 [00:00<?, ?it/s]


Full val accuracy: 0.4895
Previous best leaderboard: 0.768


## Cell 11: Submit

In [13]:
all_preds = []
for start in tqdm(range(0, len(test_df), BATCH_SIZE_INFER), desc="Test"):
    batch = test_df.iloc[start:start+BATCH_SIZE_INFER]
    try:
        p = predict_batch(batch)
    except RuntimeError:
        torch.cuda.empty_cache()
        p = []
        for j in range(len(batch)):
            p.extend(predict_batch(batch.iloc[j:j+1]))
            torch.cuda.empty_cache()
    all_preds.extend(p)
    torch.cuda.empty_cache()

submission = pd.DataFrame({"id": test_df["id"], "answer": all_preds})
submission.to_csv("/content/submission.csv", index=False)
submission.to_csv("/content/drive/MyDrive/submission_v7_clean.csv", index=False)
print(f"Saved ({len(submission)} rows)")

from google.colab import files
files.download("/content/submission.csv")

Test:   0%|          | 0/252 [00:00<?, ?it/s]

Saved (1008 rows)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>